In [66]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

In [67]:
IMPORTANCE_THRESHOLD = 0.01

RANDOM_STATE = 42

N_ESTIMATORS = 200

CLASS_WEIGHT = "balanced"

In [68]:
X_train = pd.read_pickle("../data/processed/X_train.pkl")
X_val = pd.read_pickle("../data/processed/X_val.pkl")
X_test = pd.read_pickle("../data/processed/X_test.pkl")

y_train = pd.read_pickle("../data/processed/y_train.pkl")
y_val = pd.read_pickle("../data/processed/y_val.pkl")
y_test = pd.read_pickle("../data/processed/y_test.pkl")

### Важность признаков

In [69]:
rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)

rf.fit(X=X_train, y=y_train)

rf_importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

In [70]:
selected_features = rf_importance[rf_importance > IMPORTANCE_THRESHOLD].index.tolist()

importance_df = pd.DataFrame(
    {
        "feature": X_train.columns,
        "importance": rf.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_df.head(len(selected_features))

,feature,importance
159,labs_per_day,0.064953
3,num_lab_procedures,0.060602
157,meds_per_day,0.059491
5,num_medications,0.053431
8,number_inpatient,0.046630
2,time_in_hospital,0.034983
1,age,0.034619
158,procedures_per_day,0.034290
9,number_diagnoses,0.028378
4,num_procedures,0.023318


In [71]:
print(f"Отобрано {len(selected_features)} признаков по Random Forest")

X_train[selected_features]

Отобрано 23 признаков по Random Forest


,labs_per_day,num_lab_procedures,meds_per_day,num_medications,number_inpatient,time_in_hospital,age,procedures_per_day,number_diagnoses,num_procedures,...,number_emergency,diag_3_Circulatory,medical_specialty_Other,diag_2_Circulatory,race_Caucasian,payer_code_Other,admission_source_id_7,metformin,diag_1_Circulatory,payer_code_MC
0,5.500000,-1.269231,6.500000,-0.2,1.0,-0.50,2,2.500000,-0.666667,2.0,...,0.0,0,1,0,0,1,1,0,0,0
1,22.000000,0.000000,8.000000,0.1,0.0,-0.50,3,0.500000,-0.333333,0.0,...,0.0,1,1,0,1,1,1,0,0,0
2,51.000000,0.269231,8.000000,-0.7,0.0,-0.75,4,0.000000,-1.000000,-0.5,...,0.0,0,1,0,1,1,1,0,0,0
3,10.333333,-0.500000,5.333333,0.1,0.0,-0.25,5,2.000000,0.333333,2.5,...,0.0,0,1,1,1,1,0,0,1,0
4,5.230769,0.923077,2.153846,1.3,0.0,2.25,8,0.153846,0.000000,0.5,...,0.0,0,1,1,1,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61239,4.200000,-0.884615,3.200000,0.1,1.0,0.25,7,0.200000,0.333333,0.0,...,0.0,0,1,0,1,1,1,0,0,0
61240,1.000000,-1.653846,15.000000,0.0,0.0,-0.75,8,0.000000,-0.333333,-0.5,...,0.0,0,1,0,1,0,1,0,1,1
61241,6.600000,-0.423077,3.600000,0.3,1.0,0.25,8,0.600000,0.333333,1.0,...,0.0,0,1,0,0,0,0,0,0,1
61242,4.500000,0.038462,2.100000,0.6,1.0,1.50,8,0.200000,0.333333,0.5,...,0.0,0,0,0,1,0,1,0,0,1


Обучен Random Forest для оценки важности признаков. Выводы:

- feature engineering оправдан. labs_per_day, meds_per_day, procedures_per_day вошли в топ

- diag_*_Circulatory (сопутствующие болезни кровообращения) важнее diag_1_Circulatory (основного диагноза): в топ вошли diag_2_Circulatory и diag_3_Circulatory. Пациент с диабетом и сопутствующими сердечно-сосудистыми заболеваниями вероятнее обратится в больницу снова.

- демографические и административные признаки (race_Caucasian, payer_code_Other, admission_source_id_7) - в конце топа, вклад слабый (0.01). Ожидаемо: они косвенно связаны с тяжестью пациента, но не являются прямыми предикторами.

In [72]:
X_train[selected_features].to_pickle("../data/processed/X_train_selected.pkl")
X_val[selected_features].to_pickle("../data/processed/X_val_selected.pkl")
X_test[selected_features].to_pickle("../data/processed/X_test_selected.pkl")